In [1]:
import sys
import os
sys.path.append('/home/jovyan/work')

from utils.AlinharETL import AlinharETL
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from delta import *

In [2]:
bucket = "gold"
datamart = "pessoas"
extrator = AlinharETL(bucket,datamart)

2024-10-21 09:01:29,624 - INFO - Iniciando Sessão Spark.


In [3]:
df = extrator.carrega_dados_delta('dim_pessoa')

2024-10-21 09:01:34,593 - INFO - Arquivo yaml carregada com sucesso
2024-10-21 09:01:34,594 - INFO - ['/home/jovyan/work/yaml/gold/pessoas/dim_pessoa.yaml']
2024-10-21 09:01:34,604 - INFO - Aguarde... Carregando dados
2024-10-21 09:01:45,571 - INFO - Dados carregado com sucesso


In [4]:
chars_list = ['\x00','\n','\r','\t']
df = extrator.data_quality_df(df,chars_list)
extrator.salvar_delta(df,'overwrite')

2024-10-21 09:01:46,439 - INFO - Aguarde... Persistindo dados (overwrite)
2024-10-21 09:02:19,208 - INFO - Dados persistidos com sucesso
2024-10-21 09:02:19,210 - INFO - s3a://gold/pessoas/dim_pessoa
2024-10-21 09:02:19,210 - INFO - Aguarde... Realizando optimize
2024-10-21 09:02:27,176 - INFO - Optimize executado com sucesso.
2024-10-21 09:02:27,177 - INFO - Aguarde... Realizando vacuum
2024-10-21 09:02:46,122 - INFO - Vacuum executado com sucesso.


In [5]:
extrator.conexao_banco_de_dados(os.getenv("DB_DESTINO_DBMS"),
                                os.getenv("DB_DESTINO_HOST"),
                                os.getenv("DB_DESTINO_PORT"),
                                os.getenv("DB_DESTINO_USER"),
                                os.getenv("DB_DESTINO_PASSWORD"),
                                os.getenv("DB_DESTINO_DATABASE"),
                                os.getenv("DB_DESTINO_SCHEMA"))

2024-10-21 09:02:46,128 - INFO - Aguarde... Configurando conexao banco de dados


In [6]:
extrator.persistencia_consume('pessoas_dim_pessoa')

2024-10-21 09:02:46,136 - INFO - Aguarde... Persistindo os dados camada Consume
2024-10-21 09:03:21,437 - INFO - Camada Consume persistida com sucesso
2024-10-21 09:03:21,438 - INFO - dw.pessoas_dim_pessoa


In [7]:
extrator.parar_sessao()

2024-10-21 09:03:21,973 - INFO - Sessão Spark finalizada.
